# Inspect a massive calibration run

Full remote calibrations write under `~/dev/EMU/tbmacro_data/<run_id>/`: an ArviZ `.nc`, a sampled-outputs pickle, a timestamped provenance `.log`, and a `run.log` from the job. This notebook loads those artefacts. Set `RUN_ID` to a task folder, or leave it `None` to take the most recently modified run that has a `.nc`.

Use the **tb-macro-summer4 (pixi)** kernel.


## Choose a run

`DATA_ROOT` is the shared store. Each child directory is one job (`run_id`). Inside it, files share a UTC stamp such as `20260917T1241Z`.


In [ ]:
from pathlib import Path

import arviz as az
import pandas as pd

DATA_ROOT = Path.home() / "dev" / "EMU" / "tbmacro_data"
RUN_ID: str | None = None  # e.g. "60160810", or None for the newest run


def list_run_dirs(root: Path) -> list[Path]:
    return sorted(
        (path for path in root.iterdir() if path.is_dir() and any(path.glob("*.nc"))),
        key=lambda path: path.stat().st_mtime,
    )


def resolve_run_dir(root: Path, run_id: str | None) -> Path:
    if run_id is not None:
        path = root / run_id
        if not path.is_dir():
            raise FileNotFoundError(path)
        return path
    runs = list_run_dirs(root)
    if not runs:
        raise FileNotFoundError(f"No run directories with .nc under {root}")
    return runs[-1]


def resolve_stamp(run_dir: Path) -> str:
    nc_files = sorted(run_dir.glob("*.nc"), key=lambda path: path.stat().st_mtime)
    if not nc_files:
        raise FileNotFoundError(f"No .nc under {run_dir}")
    return nc_files[-1].stem


run_dir = resolve_run_dir(DATA_ROOT, RUN_ID)
stamp = resolve_stamp(run_dir)
idata_path = run_dir / f"{stamp}.nc"
outputs_path = run_dir / f"{stamp}_outputs.pkl"
stamp_log_path = run_dir / f"{stamp}.log"
run_log_path = run_dir / "run.log"

print(f"run_id {run_dir.name}")
print(f"stamp  {stamp}")
print(f"dir    {run_dir}")


## Load artefacts

`idata` is the NUTS posterior. The pickle holds scenario outputs and sample labels written after sampling. The two logs are the job stream and the stamped provenance file that names the commit and artefacts.


In [ ]:
idata = az.from_netcdf(idata_path)
payload = pd.read_pickle(outputs_path)
outputs = payload["outputs"]
sample_labels = payload["sample_labels"]

stamp_log = stamp_log_path.read_text() if stamp_log_path.exists() else None
run_log = run_log_path.read_text() if run_log_path.exists() else None

print(f"idata   {idata_path.name}  chains={idata.posterior.sizes['chain']}  draws={idata.posterior.sizes['draw']}")
print(f"pickle  {outputs_path.name}  scenarios={len(outputs)}  samples={len(sample_labels)}")
print(f"logs    stamp={'yes' if stamp_log is not None else 'no'}  run={'yes' if run_log is not None else 'no'}")
if stamp_log is not None:
    print("--- stamp log ---")
    print(stamp_log.rstrip())
if run_log is not None:
    print("--- run log ---")
    print(run_log.rstrip())


## Posterior summary


In [ ]:
idata.posterior

In [ ]:
az.summary(idata)
